## ASSIGNMENT NLP – 3: Chatbot using Transformers
## Data Science Internship – February 2026

**Objective:** Build a simple conversational chatbot using a pre-trained transformer model from Hugging Face that can interact with users and generate meaningful responses.


## Step 1: Install Required Libraries

In [ ]:
# Install required libraries (run this cell only once)
!pip install transformers torch

2. Model Loading (Mandatory)
We will use DialoGPT-medium by Microsoft, a pre-trained model specifically fine-tuned for conversational responses.

In [ ]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer

# Define the model name from Hugging Face Model Hub [cite: 20]
model_name = "microsoft/DialoGPT-medium"

# Load the tokenizer and the pre-trained model [cite: 34, 35]
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForCausalLM.from_pretrained(model_name)

print("Model loaded successfully. You can now start the conversation!")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/642 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/614 [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

...

model.safetensors:   0%|          | 0.00/863M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/293 [00:00<?, ?it/s]

The tied weights mapping and config for this model specifies to tie transformer.wte.weight to lm_head.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning
GPT2LMHeadModel LOAD REPORT from: microsoft/DialoGPT-medium
Key                              | Status     |  | 
---------------------------------+------------+--+-
transformer.h.{0...23}.attn.bias | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


generation_config.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

Model loaded successfully. You can now start the conversation!


## 3 : Run the Chatbot

The chatbot:
- Greets the user on startup
- Accepts continuous user input
- Maintains conversation context across turns
- Exits gracefully when the user types `exit` or `quit`

In [ ]:
import warnings
import logging
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM
from transformers import logging as transformers_logging

# 1. Clean up console output and suppress warnings
warnings.filterwarnings('ignore')
transformers_logging.set_verbosity_error()
logging.getLogger("transformers").setLevel(logging.ERROR)

class QwenChatbot:
    def __init__(self, model_name="Qwen/Qwen2.5-1.5B-Instruct", system_prompt="You are a helpful AI assistant."):
        """Initializes the model, tokenizer, and starting chat history."""
        print(f"[System] Loading {model_name}... This might take a moment.")

        self.tokenizer = AutoTokenizer.from_pretrained(model_name)

        self.model = AutoModelForCausalLM.from_pretrained(
            model_name,
            torch_dtype="auto",
            device_map="auto"
        )
        self.model.eval()

        # Initialize conversation history with the system persona
        self.chat_history = [{"role": "system", "content": system_prompt}]
        print("[System] Model loaded successfully!\n" + "="*50)

    def generate_reply(self, user_message, max_new_tokens=256):
        """Processes the user input and returns the generated model response."""
        # Append user message
        self.chat_history.append({"role": "user", "content": user_message})

        # Format prompt
        prompt_text = self.tokenizer.apply_chat_template(
            self.chat_history,
            tokenize=False,
            add_generation_prompt=True
        )

        # Tokenize and move to the correct device (CPU/GPU)
        inputs = self.tokenizer([prompt_text], return_tensors="pt").to(self.model.device)

        # Generate response
        with torch.no_grad():
            output_ids = self.model.generate(
                **inputs,
                max_new_tokens=max_new_tokens,
                temperature=0.7,
                top_p=0.9,
                repetition_penalty=1.05,
                pad_token_id=self.tokenizer.eos_token_id # Explicitly define pad token
            )

        # Extract and decode only the new tokens generated by the model
        input_length = inputs["input_ids"].shape[1]
        generated_ids = output_ids[0, input_length:]

        assistant_reply = self.tokenizer.decode(generated_ids, skip_special_tokens=True).strip()

        # Save assistant reply to history
        self.chat_history.append({"role": "assistant", "content": assistant_reply})

        return assistant_reply

def main():
    # You can change the system prompt here to make the bot act differently!
    bot = QwenChatbot(
        system_prompt="You are an expert computer science tutor. Keep your answers clear, concise, and highly technical."
    )

    print("Chat session started. Type 'exit' or 'quit' to stop.")

    while True:
        try:
            user_input = input("\n[You]: ")

            # Handle empty inputs
            if not user_input.strip():
                continue

            if user_input.lower().strip() in ["exit", "quit"]:
                print("\n[Chatbot]: Shutting down. Goodbye!")
                break

            reply = bot.generate_reply(user_input)
            print(f"[Chatbot]: {reply}")

        except KeyboardInterrupt:
            # Handles the user pressing Ctrl+C gracefully
            print("\n\n[System]: Force quit detected. Goodbye!")
            break

if __name__ == "__main__":
    main()

[System] Loading Qwen/Qwen2.5-1.5B-Instruct... This might take a moment.


config.json:   0%|          | 0.00/660 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/3.09G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

[System] Model loaded successfully!
Chat session started. Type 'exit' or 'quit' to stop.
[Chatbot]: Hello! How can I assist you today with your computer science studies?
[Chatbot]: Artificial Intelligence (AI) refers to the simulation of human intelligence in machines that are programmed to think and learn like humans. AI involves the use of algorithms, statistical models, and machine learning techniques to enable computers to perform tasks that typically require human intelligence, such as visual perception, speech recognition, decision-making, and language translation.

Key components of AI include:

1. **Machine Learning**: A subset of AI where a system improves its performance on a specific task by learning from data without being explicitly programmed. This includes supervised, unsupervised, and reinforcement learning.

2. **Deep Learning**: A subfield of machine learning inspired by the structure and function of the brain known as neural networks, which are designed to recognize 

## Pipeline Flow Summary

```
User Input
    ↓
Tokenizer encodes input + appends conversation history
    ↓
DialoGPT Model generates response tokens
    ↓
Tokenizer decodes response tokens → readable text
    ↓
Display Output to User
    ↓
Loop Until User types 'exit' or 'quit'
```

## Key Concepts Used

| Concept | Description |
|---|---|
| **DialoGPT** | Pre-trained conversational model by Microsoft fine-tuned on dialogue data |
| **AutoTokenizer** | Converts raw text to token IDs compatible with the model |
| **AutoModelForCausalLM** | Causal language model for next-token prediction / text generation |
| **chat_history_ids** | Maintains multi-turn conversation context across user inputs |
| **Top-k Sampling** | Selects from top-k probable tokens for diverse, quality responses |
| **Temperature** | Controls randomness of generation (lower = more focused output) |
